## Data descriptions

---
In this competition you are predicting the probability that an online transaction is fraudulent, as denoted by the binary target isFraud.

The data is broken into two files identity and transaction, which are joined by TransactionID. Not all transactions have corresponding identity information.

Categorical Features - Transaction
ProductCD
card1 - card6
addr1, addr2
P_emaildomain
R_emaildomain
M1 - M9
Categorical Features - Identity
DeviceType
DeviceInfo
id_12 - id_38
The TransactionDT feature is a timedelta from a given reference datetime (not an actual timestamp).

You can read more about the data from this post by the competition host.

Files
train_{transaction, identity}.csv - the training set
test_{transaction, identity}.csv - the test set (you must predict the isFraud value for these observations)
sample_submission.csv - a sample submission file in the correct format

---

I see many questions regarding data description, so it maybe a better idea to open a thread for discussion. The following is a bit more details about it:

Transaction Table *

TransactionDT: timedelta from a given reference datetime (not an actual timestamp)

TransactionAMT: transaction payment amount in USD

ProductCD: product code, the product for each transaction

card1 - card6: payment card information, such as card type, card category, issue bank, country, etc.

addr: address

dist: distance

P_ and (R__) emaildomain: purchaser and recipient email domain

C1-C14: counting, such as how many addresses are found to be associated with the payment card, etc. The actual meaning is masked.

D1-D15: timedelta, such as days between previous transaction, etc.

M1-M9: match, such as names on card and address, etc.

Vxxx: Vesta engineered rich features, including ranking, counting, and other entity relations.

Categorical Features: ProductCD card1 - card6 addr1, addr2 P_emaildomain R_emaildomain M1 - M9

Identity Table *

Variables in this table are identity information – network connection information (IP, ISP, Proxy, etc) and digital signature (UA/browser/os/version, etc) associated with transactions. They're collected by Vesta’s fraud protection system and digital security partners. (The field names are masked and pairwise dictionary will not be provided for privacy protection and contract agreement)

Categorical Features: DeviceType DeviceInfo id_12 - id_38

---

In [2]:
import pandas as pd
import networkx as nx

In [3]:
train_identity = pd.read_csv('ieee-fraud-detection/train_identity.csv')
train_transaction = pd.read_csv('ieee-fraud-detection/train_transaction.csv')


In [4]:
train_identity.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [17]:
train_transaction.head().iloc[:, :20]


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,19.0,NaN,NaN,NaN,1.0,1.0,0.0
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0,87.0,287.0,NaN,outlook.com,NaN,1.0,1.0,0.0
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,87.0,NaN,NaN,yahoo.com,NaN,2.0,5.0,0.0
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0


# Plan for making a graph
- Many nodes: eamils, transaction id, addresses, devices, IP information (identify all of these in the two data tables and work on relating the in networkx)
- Make all edges connections from transaction to other node types

``` 
Basic example to build from
import networkx as nx

G = nx.Graph()

for _, row in df.iterrows():
    t = f"T_{row['TransactionID']}"
    card = f"C_{row['card1']}"
    
    G.add_edge(t, card)
```

In [38]:
train_identity[['id_01', 'id_02']]

,id_01,id_02
0,0.0,70787.0
1,-5.0,98945.0
2,-5.0,191631.0
3,-5.0,221832.0
4,0.0,7460.0
...,...,...
144228,-15.0,145955.0
144229,-5.0,172059.0
144230,-20.0,632381.0
144231,-5.0,55528.0


In [ ]:
# get all connections created
# zipping the target cols as string and using add_edges_from() is more efficent so use this

# handle this for all of the transaction data first and then handle it for the identityt data

transactions_graph = nx.Graph()

# target cols
# all card data (card1 - card6)
card_cols = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6']

for col in card_cols:
    # need to handle the Na data, so maksing before adding anything to the graph
    mask = train_transaction[col].notna()

    target_data = train_transaction[mask].copy()

    transaction_card_attribute_connection = zip(
        'Tr' + target_data.TransactionID.astype('str'), 
        f'C{col[-1]}' + target_data[col].astype('str') # puts creates prefix of C1, C2, ... to C6 so each card identifier is unique
    )

    # add it to the graph
    transactions_graph.add_edges_from(transaction_card_attribute_connection)
    



done


In [ ]:
def create_graph_edges_for_similar_cols(
        target_graph: nx.Graph,
        target_df: pd.DataFrame, 
        primary_node_col: str,
        primary_node_prefix: str,
        target_col_names: list[str], 
        graph_label_prefix: str, 
        graph_labels_suffix: list[str] = None
    ) -> None:
    """
        This is a utility function for iterating similar columns that might need to be added to a graph in bulk. It will edit the graph passed in the argument directly.

        graph: An instance of a networkx graph that needs to have connections added to it
        target_df: The data frame containing all reference columns for the mappings
        primary_node_col: The name of the column that will server as the primary node that should as the basis for many connections (ex. Transactions in a case where you want to connect many other attributes about that transaction to it),
        primary_node_prefix: A string that is the prefix for the primary node that will be used on the Graph (ex. if the primary node is Transactions you may want to use something like 'T' or 'Tr')
        target_col_names: a list of strings containing the names of columns in the target DataFrame that need to have relationships created for
        graph_label_prefix: the desired prefix for the target column to be associated with in the graph (ex if the columns are all related to credit card info you may want to use somehting like 'C')
        graph_label_suffix: this is the method for appending unique identifiers to the graph_label_prefix when there are multiple cols (once again for card info you may want something like C1, C2 ...)
                            If nothing is passed to this argument it will fill with 1 - n where n is the len of the  target_col_names. If you want custom functionality pass list of the suffix values in order.
    """
    tdf = target_df.copy() # ensure that the orginal df is not edited by this

    if graph_labels_suffix != None:
        assert len(set(graph_labels_suffix)) == len(graph_labels_suffix), 'Graph label suffix values must be unique.'
    else: 
        graph_labels_suffix = list(range(1, len(graph_label_prefix) + 1)) # default functionality for using digits as the suffix

    for col_name, suffix in (target_col_names, graph_labels_suffix):
        mask = tdf[col_name].notna()
        
        target_data = tdf[mask]

        connection_tuple_mappings = zip(
            f'{primary_node_prefix}' + target_data[primary_node_col].astype('str'), 
            f'{graph_label_prefix}{suffix}' + target_data[col].astype('str') # puts creates prefix of C1, C2, ... to C6 so each card identifier is unique
        )

        target_graph.add_edges_from(connection_tuple_mappings)
    


In [ ]:
# train_transaction[train_transaction[['card1', 'card2', 'card3', 'card4', 'card5', 'card6']].notna()]
print('done')

# address info (addr1 and addr2 )

# product code

3525335

[range(1, 10)]